# MP 5: Diffusion Transformers

*Created by Jay Mahajan & Ozgur Kara (Spring 2026 CS 444 TAs)*

You will be implementing a custom Diffusion Transformer. Your goal is to implement a transformer (Part A) and then use it to generate handwritten digits (Part B).

Files that need to be edited: ```DiT.py```

In [ ]:
!pip install torch torchvision matplotlib tqdm einops torchmetrics[image] torch-fidelity diffusers


In [ ]:
# Run only on Colab
import os
try:
  from google.colab import drive
  drive.mount('/content/drive/')
  os.chdir('/content/drive/MyDrive/<YOUR FOLDER>')
except:
  print("Not Running on Colab")


## Part A: Implementing the transformer

You need to implement the transformer in ```DiT.py```. You will need to follow the instructions in the website.

For each sub part, you can test if it works by running ```python tests.py```. **You must pass all tests in order to get full credit!**

In [ ]:
!python tests.py

## Part B: Training with DDPM

Once you have the transformer built, you will train it with DDPM.

In [ ]:
import torch
import torch.nn as nn
from DiT import DiT
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torchvision.utils import make_grid
from torchvision.transforms import functional as F
import matplotlib.pyplot as plt
from torchmetrics.image.fid import FrechetInceptionDistance
%matplotlib inline

In [ ]:
### Set Hyperparameters, do not change these, unless you are doing extra credit
res = 28
patch_size = 4
channels = 1
heads = 8
hidden_dim = 256
num_patch = (res // patch_size) ** 2
ff_dim = 1024
time_emb_dim = 256
num_blocks = 5
num_timesteps = 300

beta_0 = 0.0001
beta_t = 0.02

batch_size = 64
num_epoch = 100
lr=4e-4

In [ ]:
### Helper function to show images
def show_images(imgs):
    batch_min = imgs.view(imgs.size(0), -1).min(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
    batch_max = imgs.view(imgs.size(0), -1).max(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
    imgs = (imgs - batch_min) / (batch_max - batch_min + 1e-6)
    imgs = imgs.detach().cpu()
    grid = make_grid(imgs, nrow=5)
    plt.axis('off')
    plt.imshow(grid.permute(1, 2, 0))
    plt.show()

In [ ]:
import os

class Diffusion:

    def __init__(self):
        trans = transforms.Compose([
            transforms.Resize(res),
            transforms.ToTensor(),
            transforms.Normalize((0.5), (0.5)),
        ])
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.fid = FrechetInceptionDistance(feature=2048, normalize=True).to(self.device)
        self.num_fid_samples = 1000

        data_root = os.path.join(os.getcwd(), "mnist_data")
        self.dataset = datasets.MNIST(
            root=data_root,
            train=True,
            download=True,
            transform=trans,
        )
        self.training_dataloader = DataLoader(
            self.dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available(),
        )

        self.betas = torch.linspace(beta_0, beta_t, num_timesteps, device=self.device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)

        self.model = DiT(
            patch_size=patch_size,
            num_blocks=num_blocks,
            num_heads=heads,
            ff_dim=ff_dim,
            time_emb_dim=time_emb_dim,
            num_timesteps=num_timesteps,
            hidden_dim=hidden_dim,
            num_patches=num_patch,
            num_channels=channels,
        ).to(self.device)

        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.criterion = nn.MSELoss()

        self._precompute_fid_real()

    def train(self):
        total_losses = []
        all_fids = []
        fid_epochs = []
        for e in range(num_epoch):
            self.model.train()

            all_losses = []

            for imgs, _ in tqdm(self.training_dataloader):
                imgs = imgs.to(self.device)
                b = imgs.shape[0]
                t = torch.randint(0, num_timesteps, (b,), device=self.device, dtype=torch.long)
                noise = torch.randn_like(imgs)
                ab = self.alphas_cumprod[t].view(-1, 1, 1, 1)
                x_t = torch.sqrt(ab) * imgs + torch.sqrt(1.0 - ab) * noise
                pred = self.model(x_t, t)
                loss = self.criterion(pred, noise)
                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                self.optimizer.step()
                all_losses.append(loss.item())

            print("Avg Loss:", sum(all_losses) / len(all_losses))
            total_losses += all_losses
            all_losses = []
            if e % 10 == 0:
                print("Epoch:", e + 1)
                imgs = self.run_inference()
                fid_val = self.compute_fid()
                print("FID:", fid_val)
                all_fids.append(fid_val)
                fid_epochs.append(e + 1)
                show_images(imgs)

        print("Final Epoch:")
        final_imags = self.run_inference()
        show_images(final_imags)

        plt.plot(total_losses)
        plt.title("Loss")
        plt.xlabel("Steps")
        plt.show()

        plt.plot(fid_epochs, all_fids)
        plt.title("FID")
        plt.xlabel("Epochs")
        plt.show()

    def run_inference(self, num_samples=20):
        self.model.eval()

        with torch.no_grad():
            x = torch.randn(
                num_samples, channels, res, res, device=self.device, dtype=torch.float32
            )
            for t in reversed(range(num_timesteps)):
                t_batch = torch.full(
                    (num_samples,), t, device=self.device, dtype=torch.long
                )
                eps = self.model(x, t_batch)
                alpha_t = self.alphas[t].view(1, 1, 1, 1)
                beta_t = self.betas[t].view(1, 1, 1, 1)
                ab_t = self.alphas_cumprod[t].view(1, 1, 1, 1)
                mean = (1.0 / torch.sqrt(alpha_t)) * (
                    x - (beta_t / torch.sqrt(1.0 - ab_t)) * eps
                )
                if t > 0:
                    z = torch.randn_like(x)
                    sigma = torch.sqrt(beta_t)
                    x = mean + sigma * z
                else:
                    x = mean
            return x

    def _prepare_for_fid(self, imgs_m1_1):
        imgs = (imgs_m1_1 / 2 + 0.5).clamp(0, 1)
        imgs = imgs.repeat(1, 3, 1, 1)
        imgs = F.resize(imgs, [75, 75], antialias=True)
        return imgs

    def _precompute_fid_real(self):
        print(
            f"[FID] Pre-computing real features on up to {self.num_fid_samples} samples...")
        total = 0
        for imgs, _ in self.dataset:
            imgs = imgs.to(self.device)
            if imgs.dim() == 3:
                imgs = imgs.unsqueeze(0)
            imgs = self._prepare_for_fid(imgs)
            self.fid.update(imgs, real=True)
            total += imgs.size(0)
            if total >= self.num_fid_samples:
                break
        print(f"[FID] Real features ready ({total} samples).")

    @torch.no_grad()
    def compute_fid(self):
        if self.fid is None:
            return None
        self.model.eval()
        batch = 200
        needed = self.num_fid_samples
        done = 0

        while done < needed:
            n = min(batch, needed - done)
            fake = self.run_inference(num_samples=n)
            fake = self._prepare_for_fid(fake.to(self.device))
            self.fid.update(fake, real=False)
            done += n

        val = float(self.fid.compute().item())
        self.fid.reset()
        self._precompute_fid_real()
        self.model.train()
        return val


In [ ]:
d = Diffusion()
d.train()

## Extra Credit: Complete any extra credit here